# Population Cortical Depth Profile CI/CV QC

This notebook evaluates regional cortical depth profiles across the cohort using the mean profile, 95% confidence interval, standard deviation, and coefficient of variation. It processes one CSV at a time and does not clean, interpolate, delete, or overwrite profiles.

In [ ]:
from pathlib import Path
import re
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/results/column_analysis')
QC_ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/results/column_quality_qc')
QC_ROOT.mkdir(parents=True,exist_ok=True)
PATTERN=re.compile(r'^(?P<subject>D\d+)_(?P<hemi>lh|rh)_(?P<region>.+)_(?P<metric>md|fa|ad|rd)\.csv$')
DEPTHS=21
print('Reading:',RESULTS_ROOT)
print('Writing:',QC_ROOT)

## 1. Regional CI/CV summaries

For each subject × hemisphere × region × metric, trajectories are summarized at every cortical depth. The 95% CI is calculated as `mean ± 1.96 × SD / sqrt(n)`, and CV is `SD / abs(mean)`.

In [ ]:
rows=[]
for profile_dir in sorted(RESULTS_ROOT.glob('D*/profiles_before_clean')):
    for path in sorted(profile_dir.glob('*.csv')):
        m=PATTERN.match(path.name)
        if not m: continue
        meta=m.groupdict()
        base={**meta,'path':str(path)}
        if path.stat().st_size==0:
            rows.append({**base,'status':'empty','n_trajectories':0,'n_depths':0,'mean_value':np.nan,'mean_sd':np.nan,'mean_ci_width':np.nan,'mean_cv':np.nan})
            continue
        try:
            a=np.loadtxt(path,delimiter=',',ndmin=2)
            if a.shape[1] != DEPTHS:
                rows.append({**base,'status':'unexpected_depth_count','n_trajectories':a.shape[0],'n_depths':a.shape[1],'mean_value':np.nan,'mean_sd':np.nan,'mean_ci_width':np.nan,'mean_cv':np.nan})
            else:
                finite=np.isfinite(a)
                means=np.nanmean(np.where(finite,a,np.nan),axis=0)
                sd=np.nanstd(np.where(finite,a,np.nan),axis=0,ddof=1)
                n=np.sum(finite,axis=0)
                ci_width=2*1.96*sd/np.sqrt(np.maximum(n,1))
                cv=sd/np.abs(means)
                rows.append({**base,'status':'valid','n_trajectories':a.shape[0],'n_depths':a.shape[1],'mean_value':float(np.nanmean(means)),'mean_sd':float(np.nanmean(sd)),'mean_ci_width':float(np.nanmean(ci_width)),'mean_cv':float(np.nanmean(cv))})
            del a
            gc.collect()
        except Exception as exc:
            rows.append({**base,'status':'unreadable','n_trajectories':0,'n_depths':0,'mean_value':np.nan,'mean_sd':np.nan,'mean_ci_width':np.nan,'mean_cv':np.nan,'error':str(exc)})
summary=pd.DataFrame(rows)
summary.to_csv(QC_ROOT/'regional_ci_cv_summary.csv',index=False)
print('Profiles:',len(summary),'Subjects:',summary.subject.nunique())
display(summary.head())

## 2. Population summaries

These tables summarize profile precision and relative trajectory variability by subject, metric, hemisphere, and region.

In [ ]:
def aggregate(group):
    return group.agg(n_profiles=('path','size'),n_valid_profiles=('status',lambda s:int((s=='valid').sum())),n_empty_profiles=('status',lambda s:int((s=='empty').sum())),mean_ci_width=('mean_ci_width','mean'),median_ci_width=('mean_ci_width','median'),mean_cv=('mean_cv','mean'),median_cv=('mean_cv','median'),mean_sd=('mean_sd','mean')).reset_index()
subject_summary=aggregate(summary.groupby('subject'))
metric_summary=aggregate(summary.groupby('metric'))
hemisphere_summary=aggregate(summary.groupby(['subject','hemi','metric']))
region_summary=aggregate(summary.groupby(['subject','hemi','region','metric']))
for name,df in [('subject',subject_summary),('metric',metric_summary),('subject_hemisphere_metric',hemisphere_summary),('subject_hemisphere_region_metric',region_summary)]: df.to_csv(QC_ROOT/f'{name}_ci_cv_summary.csv',index=False)
display(subject_summary); display(metric_summary)

## 3. Group-level regional rankings

Regions are ranked by mean CV and mean 95% CI width across all available subjects and hemispheres. Lower values indicate more consistent trajectories or a more precise regional mean estimate.

In [ ]:
ranking=(summary[summary.status=='valid'].groupby(['metric','region'],as_index=False).agg(subjects=('subject','nunique'),profiles=('path','size'),mean_cv=('mean_cv','mean'),mean_ci_width=('mean_ci_width','mean'),mean_sd=('mean_sd','mean'),mean_value=('mean_value','mean')).sort_values(['metric','mean_cv']))
ranking.to_csv(QC_ROOT/'population_region_rankings.csv',index=False)
for metric in sorted(ranking.metric.unique()):
    print(f'===== {metric.upper()} =====')
    display(ranking[ranking.metric==metric].head(10))
    display(ranking[ranking.metric==metric].sort_values('mean_cv',ascending=False).head(10))

## 4. Population visualizations

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
valid=summary[summary.status=='valid']
valid.boxplot(column='mean_cv',by='metric',ax=axes[0]); axes[0].set_title('Regional CV across the cohort'); axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Mean CV'); plt.sca(axes[0]); plt.suptitle('')
valid.boxplot(column='mean_ci_width',by='metric',ax=axes[1]); axes[1].set_title('Regional 95% CI width across the cohort'); axes[1].set_xlabel('Metric'); axes[1].set_ylabel('Mean CI width'); plt.sca(axes[1]); plt.suptitle('')
metric_plot=valid.groupby(['subject','metric'],as_index=False).mean_cv.mean().reset_index() if False else valid.groupby(['subject','metric'],as_index=False)['mean_cv'].mean()
for metric,g in metric_plot.groupby('metric'): axes[2].plot(g.subject,g.mean_cv,marker='o',label=metric.upper())
axes[2].set_title('Subject-level mean CV'); axes[2].set_xlabel('Subject'); axes[2].set_ylabel('Mean CV'); axes[2].tick_params(axis='x',rotation=90); axes[2].legend(); axes[2].grid(alpha=.25)
fig.tight_layout(); fig.savefig(QC_ROOT/'population_ci_cv_overview.png',dpi=180); display(fig)